# 06 · Conversational liveness  (P5, Branch D)

A different question from the rest of the system: not *was this waveform
manufactured* but **is a machine in the conversational loop**.

The design decisions worth keeping in mind while working here:

- **Shape, never the mean.** A poor connection also adds 300 ms. Network
  latency shifts the whole distribution uniformly; it does not impose a hard
  floor, remove overlaps, or compress variance.
- **Within-call comparison.** The agent side is a known human, so it is the
  reference. This cancels network, task, register, speaker pair — and
  language, which matters because turn-taking gaps vary widely across
  languages and there is no data at all for Indian telephone registers.
- **The entire feature set is timestamps.** No audio, no embeddings. Which is
  why this branch can run where the acoustic branch legally cannot.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q pydantic pyyaml numpy matplotlib

## Validate against ground truth first

Before touching real recordings, confirm the tracker recovers a conversation
whose timing we control. The synthetic adapter can impose a hard floor on the
caller side, which is exactly what a buffered voice converter does.

In [ ]:
import asyncio
from vif.common.config import load_config
from vif.common.types import Side
from vif.serve.adapters.file import SyntheticAdapter
from vif.serve.detector import StubDetector
from vif.serve.session import CallSession
from vif.serve.vad import EnergyVAD

config = load_config("configs")
config.model.liveness.enabled = True  # off by default; this notebook is about it

async def run_call(floor_ms, seed=1, turns=18):
    adapter = SyntheticAdapter(n_turns=turns, pipeline_floor_ms=floor_ms,
                               seed=seed, realtime=False)
    session = CallSession(
        session_id=f"floor{floor_ms}", config=config,
        detector=StubDetector(config.model.audio.window_samples),
        vad=EnergyVAD(config.model.audio.vad_frame),
        )
    session.liveness.set_rtt(await adapter.rtt_ms())
    await asyncio.gather(
        session.consume(adapter, Side.CALLER),
        session.consume(adapter, Side.AGENT),
    )
    payload = session.finalise()
    session.close()
    return payload

human   = await run_call(0.0)
machine = await run_call(320.0)

for name, p in [("human", human), ("machine", machine)]:
    f = p.liveness
    print(f"{name:<9} transitions={f.n_transitions:3d}  "
          f"floor={f.caller_floor_ms:7.1f}ms  "
          f"overlap={0 if f.caller_overlap_rate is None else f.caller_overlap_rate:.2f}  "
          f"llr={p.liveness and 'see features'}")

The detected caller floor should land close to the 320 ms we imposed. If it
does not, the turn tracker is mis-segmenting and every number from this branch
is suspect.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from vif.serve.liveness import build_transitions

def gaps_for(payload_session_floor, seed=1):
    return payload_session_floor

fig, ax = plt.subplots(figsize=(8, 4))
for label, floor, colour in [("human", 0.0, "#0C7180"), ("machine in loop", 320.0, "#B8873C")]:
    adapter = SyntheticAdapter(n_turns=40, pipeline_floor_ms=floor, seed=5, realtime=False)
    from vif.serve.liveness import Utterance
    utts = [Utterance(Side(s), a, b) for s, a, b in adapter.ground_truth()]
    trans = build_transitions(utts, config.model.liveness)
    caller_gaps = [t.gap_ms for t in trans if t.to_side == Side.CALLER]
    ax.hist(caller_gaps, bins=22, alpha=.6, label=label, color=colour)

ax.axvline(0, ls="--", c="k", lw=1)
ax.text(4, ax.get_ylim()[1]*.9, "overlap  <-  |  ->  gap", fontsize=9)
ax.set_xlabel("caller response gap (ms)")
ax.set_ylabel("count")
ax.set_title("The floor a buffered pipeline cannot go below")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Collect the real corpus

No public dataset exists, and you do not need one. Two laptops and an
afternoon:

```
Condition A   human <-> human                        (control)
Condition B   human <-> human + streaming VC in loop (the attack)
Condition C   human <-> pre-recorded playback        (the easy case)
```

Around 30 conversations per condition. Recorded with:

```
PYTHONPATH=src python scripts/collect_turntaking.py --condition A --label pair01
```

Only timestamps are written — no audio — which is both the point of the branch
and what makes the collection ethically simple.

In [ ]:
import json
from pathlib import Path
from vif.serve.liveness import TurnTracker, build_transitions, compute_features
from vif.common.types import TurnEvent, TurnEventKind

recordings = sorted(Path("data/turntaking").glob("*.json"))
print(f"{len(recordings)} recordings found")

rows = []
for path in recordings:
    blob = json.loads(path.read_text(encoding="utf-8"))
    tracker = TurnTracker()
    for e in blob["events"]:
        tracker.record(TurnEvent(side=Side(e["side"]),
                                 kind=TurnEventKind(e["kind"]),
                                 monotonic_ns=e["monotonic_ns"]))
    last = max((e["monotonic_ns"] for e in blob["events"]), default=0)
    tracker.close(last / 1e9)
    f = compute_features(build_transitions(tracker.utterances, config.model.liveness),
                         config.model.liveness)
    rows.append({"condition": blob["condition"], **f.model_dump()})

if rows:
    import pandas as pd
    df = pd.DataFrame(rows)
    print(df.groupby("condition")[
        ["floor_delta_ms", "overlap_delta", "variance_ratio", "n_transitions"]
    ].mean().round(3))

## The experiment that justifies the branch

Replay every recording with synthetic network delay injected, and confirm the
**shape** features still separate the conditions where a naive mean-latency
threshold does not. That contrast is the whole argument.

In [ ]:
import numpy as np

def shift_all_gaps(events, delay_ms, jitter_ms=40.0, seed=0):
    """Add network delay to one side, as a bad connection would."""
    rng = np.random.default_rng(seed)
    out = []
    for e in events:
        extra = 0
        if e["side"] == "caller":
            extra = int((delay_ms + rng.normal(0, jitter_ms)) * 1e6)
        out.append({**e, "monotonic_ns": e["monotonic_ns"] + extra})
    return out

# For each recording, compare a mean-latency rule against the shape features
# before and after injecting 250 ms of delay.  The mean-latency rule should
# start misclassifying; the shape features should not.
print("Run this once you have recordings in data/turntaking/.")

## Honest power calculation

Do this on real data before quoting anything.

If humans produce a sub-150 ms response at rate *p*, observing none after *n*
transitions has probability (1−*p*)ⁿ under the human hypothesis. At *p* ≈ 0.15,
ten transitions gives about 0.20 — suggestive, not conclusive. Twenty gives
about 0.04.

**So timing evidence alone plausibly needs 10–20 turns.** Do not let this be
pitched as a fast detector. It is fast in exactly two cases, and those are the
ones to demo:

- the first transition, if the offset is large against measured RTT
- pre-recorded playback, which has no turn-taking behaviour at all and
  collapses on turn one

In [ ]:
p_fast = 0.15
for n in (5, 10, 15, 20, 30):
    print(f"  {n:2d} transitions  ->  P(no fast response | human) = {(1-p_fast)**n:.3f}")